# MobilityDuck and DuckDB on the AIS data of the book (Chapter 7)

Every listing of Chapter 7 that reads the AIS data, in the order of the chapter, with the answer the chapter prints as a comment. Set before starting Jupyter:

- `AIS_DIR`: the directory of `aisdk-2026-01-20.csv` (`ais/fetch_data.sh day` of this repository downloads its archive);
- `PG_AIS`: the connection string of the AIS database built by `ais/build.sh` (for example `dbname=ais`);
- `MOBILITYDUCK_EXTENSION`: the path of `mobilityduck.duckdb_extension`.

The notebook runs DuckDB v1.5.6, as the README of this chapter explains.

In [1]:
import os

import duckdb
import pandas as pd

AIS_DIR = os.environ['AIS_DIR']
PG_AIS = os.environ['PG_AIS']
MOBILITYDUCK_EXTENSION = os.environ['MOBILITYDUCK_EXTENSION']
print(duckdb.__version__)

# The listings create their tables as the chapter prints them, so every run starts
# from a new database file
for f in ('ais_ch7.duckdb', 'ais_ch7.duckdb.wal'):
    if os.path.exists(f):
        os.remove(f)
con = duckdb.connect('ais_ch7.duckdb', config={'allow_unsigned_extensions': 'true'})
con.execute('INSTALL spatial')
con.execute('LOAD spatial')


def run(sql, rows=20):
    """Run the statements of a listing and show the first rows of the answer of the
    last one; only these rows are fetched, so that an answer of millions of rows, such
    as that of SELECT * EXCLUDE (SOG) FROM AISInput, stays in DuckDB."""
    sql = (sql.replace("'aisdk-2026-01-20.csv'", f"'{AIS_DIR}/aisdk-2026-01-20.csv'")
              .replace("'/path/to/mobilityduck.duckdb_extension'", f"'{MOBILITYDUCK_EXTENSION}'")
              .replace("'{conn_str}'", f"'{PG_AIS}'"))
    rel = con.execute(sql)
    if not rel.description:
        return None
    return pd.DataFrame(rel.fetchmany(rows), columns=[d[0] for d in rel.description])

1.5.6


## DuckDB: loading the AIS reports and SQL features

The listings of Sect. "DuckDB" on the raw AIS reports of January 20, 2026, loaded from the file `aisdk-2026-01-20.csv` of the Danish Maritime Authority (Appendix "The AIS Database of the Book").

In [2]:
run("""
SET geometry_always_xy = TRUE;
CREATE TABLE AISInput AS
  SELECT *, ST_Point(Longitude, Latitude) AS Geom
  FROM read_csv('aisdk-2026-01-20.csv', header = TRUE,
    timestampformat = '%d/%m/%Y %H:%M:%S', names = ['T', 'TypeOfMobile',
    'MMSI', 'Latitude', 'Longitude', 'NavigationalStatus', 'ROT', 'SOG', 'COG', 'Heading',
    'IMO', 'CallSign', 'Name', 'ShipType', 'CargoType', 'Width', 'Length',
    'TypeOfPositionFixingDevice', 'Draught', 'Destination', 'ETA', 'DataSourceType',
    'SizeA', 'SizeB', 'SizeC', 'SizeD']);
""")

,Count
0,17630815


In [3]:
run("""
SELECT * EXCLUDE (SOG) FROM AISInput;
""")

,T,TypeOfMobile,MMSI,Latitude,Longitude,NavigationalStatus,ROT,COG,Heading,IMO,...,TypeOfPositionFixingDevice,Draught,Destination,ETA,DataSourceType,SizeA,SizeB,SizeC,SizeD,Geom
0,2026-01-20,Class A,246483000,56.245220,12.378778,Under way using engine,0.0,167.9,169.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00z\xe3\xa40\xef\xc1(@\x9e...
1,2026-01-20,Base Station,2190064,56.716515,11.519045,Unknown value,NaN,NaN,NaN,Unknown,...,GPS,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,"b""\x01\x01\x00\x00\x00\xfdM(D\xc0\t'@\xf6\x0bv..."
2,2026-01-20,Class A,219031856,55.475085,8.421717,Under way using engine,0.0,267.0,242.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\x9fUfJ\xeb\xd7 @\xfb\xe...
3,2026-01-20,Class A,246478000,57.422855,11.618723,Under way using engine,1.1,322.1,319.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,"b""\x01\x01\x00\x00\x00\xe6\x90\xd4B\xc9<'@\x9c..."
4,2026-01-20,Class A,246483000,56.245220,12.378778,Under way using engine,0.0,167.9,169.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00z\xe3\xa40\xef\xc1(@\x9e...
5,2026-01-20,Class A,246478000,57.422855,11.618723,Under way using engine,1.1,322.1,319.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,"b""\x01\x01\x00\x00\x00\xe6\x90\xd4B\xc9<'@\x9c..."
6,2026-01-20,Base Station,2194006,55.538870,5.033200,Unknown value,NaN,NaN,NaN,Unknown,...,Surveyed,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\xa7\xe8H.\xff!\x14@\xd4...
7,2026-01-20,Class A,266464000,91.000000,0.000000,Unknown value,NaN,NaN,NaN,9816529,...,GPS,7.2,DKKAL,2026-01-20 23:00:00,AIS,94.0,26.0,3.0,16.0,b'\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00...
8,2026-01-20,Class A,219005313,56.152330,10.251313,Under way using engine,NaN,301.3,NaN,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\x8a\x1e\xf8\x18\xac\x80...
9,2026-01-20,Class A,273277840,54.940510,10.890440,Constrained by her draught,5.4,50.8,54.0,Unknown,...,Undefined,NaN,Unknown,NaT,AIS,NaN,NaN,NaN,NaN,b'\x01\x01\x00\x00\x00\xb9\x19n\xc0\xe7\xc7%@\...


In [4]:
run("""
SELECT MAX(COLUMNS('L.*')) FROM AISInput;
/* | Latitude | Longitude | Length |
   |  double  |  double   | int64  |
   |   91.0   | 40.337515 |   400  | */
""")

,Latitude,Longitude,Length
0,91.0,40.337515,400


In [5]:
run("""
SELECT SOG > 0.5 AS IsMoving, date_trunc('month', T) AS ReportMonth,
  AVG(ROT) AS AvgRot
FROM AISInput
WHERE IsMoving = TRUE
GROUP BY IsMoving, ReportMonth HAVING AvgRot > 0.01;
/* | IsMoving  |  ReportMonth | AvgRot  |
   | 258656000 | 2026-01-01 00:00:00 |  0.0155.. |  */
""")

,IsMoving,ReportMonth,AvgRot
0,True,2026-01-01,0.012871


In [6]:
run("""
SELECT MMSI, date_trunc('day', T) AS ReportDay, round(SUM(SOG) / 24, 2) AS
  TotalSogPerDay, round(AVG(COG), 2) AS AvgCogPerDay
FROM AISInput
GROUP BY ALL;
/* | MMSI      | ReportDay  | TotalSogPerDay    | AvgCogPerDay |
   | 258656000 | 2026-01-20 |  0.41 | NULL   |  247.18| */
""")

,MMSI,ReportDay,TotalSogPerDay,AvgCogPerDay
0,211325510,2026-01-20,11866.25,267.02
1,255806257,2026-01-20,31548.75,182.16
2,2655120,2026-01-20,NaN,NaN
3,244083000,2026-01-20,661.55,169.62
4,219026172,2026-01-20,725.64,29.87
5,219166000,2026-01-20,2.26,166.54
6,265609960,2026-01-20,0.35,269.34
7,219022000,2026-01-20,2195.56,186.72
8,244641000,2026-01-20,1204.55,187.95
9,266128000,2026-01-20,930.32,152.34


In [7]:
run("""
SELECT *
FROM (
  SELECT MMSI, date_trunc('day', T) AS ReportDay,  ShipType, SOG
  FROM AISInput
  WHERE MMSI = 258656000)
PIVOT (AVG(SOG) FOR ShipType IN ('Cargo', 'Tanker', 'Passenger', 'Fishing') )
ORDER BY ReportDay;
/* | MMSI      | ReportDay  | Cargo    | Tanker | Passenger | Fishing |
   | 258656000 | 2026-01-20 | 5.855474 | NULL   | 0.003226  | NULL    | */
""")

,MMSI,ReportDay,Cargo,Tanker,Passenger,Fishing
0,258656000,2026-01-20,5.855474,None,0.003226,None


In [8]:
run("""
SELECT *
FROM (
  SELECT MMSI, IF(hour(T) < 12, 'AM', 'PM') AS HalfDay, SOG
  FROM AISInput )
PIVOT (
  round(SUM(SOG)) AS total,           -- total SOG per period
  MAX(SOG) AS bestSOG                 -- max SOG per period
  FOR HalfDay IN ('AM', 'PM')         -- pivot by half of the day
)
ORDER BY MMSI;
/* | MMSI    | AM_total | AM_bestSOG | PM_total | PM_bestSOG |
   | 3638    | 0.0      | 0.0        | 0.0      | 0.0        |
   | 261505  | NULL     | NULL       | 595.0    | 60.0       |
   | 2190045 | 1.0      | 0.1        | 0.0      | 0.0        |
   ... */
""")

,MMSI,AM_total,AM_bestSOG,PM_total,PM_bestSOG
0,3638,0.0,0.0,0.0,0.0
1,261505,NaN,NaN,595.0,60.0
2,2190045,1.0,0.1,0.0,0.0
3,2190047,NaN,NaN,NaN,NaN
4,2190048,NaN,NaN,NaN,NaN
5,2190049,NaN,NaN,NaN,NaN
6,2190051,NaN,NaN,NaN,NaN
7,2190052,NaN,NaN,NaN,NaN
8,2190064,NaN,NaN,NaN,NaN
9,2190065,NaN,NaN,NaN,NaN


## Querying DuckDB

The queries of Sect. "Querying DuckDB" on the table `AISInput`.

*Count the number of records in the input dataset.*

In [9]:
run("""
SELECT COUNT(*) FROM AISInput;
-- 17630815
""")

,count_star()
0,17630815


*Number of messages per vessel.*

In [10]:
run("""
SELECT MMSI, COUNT(*) AS MessageCount
FROM AISInput
GROUP BY MMSI ORDER BY MessageCount DESC;
""")

,MMSI,MessageCount
0,2579999,101899
1,2190049,72461
2,2190068,61863
3,2190076,57390
4,2190048,53670
5,2190051,53658
6,3638,53416
7,219022903,53045
8,2190073,51655
9,2190067,51439


*First and last seen timestamps per vessel.*

In [11]:
run("""
SELECT MMSI, MIN(t) AS FirstSeen, MAX(t) AS LastSeen
FROM AISInput
GROUP BY MMSI ORDER BY LastSeen;
""")

,MMSI,FirstSeen,LastSeen
0,246993000,2026-01-20 00:00:00,2026-01-20 00:00:00
1,246440000,2026-01-20 00:00:04,2026-01-20 00:00:34
2,305078000,2026-01-20 00:01:34,2026-01-20 00:01:34
3,211290200,2026-01-20 00:02:03,2026-01-20 00:04:31
4,257643000,2026-01-20 00:05:00,2026-01-20 00:05:00
5,314001119,2026-01-20 00:00:34,2026-01-20 00:10:53
6,210617000,2026-01-20 00:00:04,2026-01-20 00:11:40
7,44133113,2026-01-20 00:13:43,2026-01-20 00:13:43
8,992651036,2026-01-20 00:17:22,2026-01-20 00:17:22
9,255915667,2026-01-20 00:00:20,2026-01-20 00:18:03


*Average an maximum speed over ground (SOG) per vessel.*

In [12]:
run("""
SELECT  MMSI, AVG(SOG) AS AvgSpeed, MAX(SOG) AS MaxSpeed
FROM AISInput
GROUP BY MMSI ORDER BY AvgSpeed DESC;
""")

,MMSI,AvgSpeed,MaxSpeed
0,226786696,212.000000,212.0
1,111257012,120.850000,150.0
2,111265555,116.584615,136.0
3,111211510,112.889916,140.0
4,111265581,103.000000,105.0
5,111219507,101.464695,102.2
6,111211512,99.639246,137.0
7,111261503,93.514825,122.0
8,111219513,91.562525,102.2
9,111219520,85.813205,102.2


*Active hours per vessel.*

In [13]:
run("""
SELECT MMSI, COUNT(DISTINCT date_trunc('hour', t)) AS ActiveHours
FROM AISInput
GROUP BY MMSI ORDER BY ActiveHours DESC;
""")

,MMSI,ActiveHours
0,265828500,24
1,219004668,24
2,265570730,24
3,219021235,24
4,265507770,24
5,219019804,24
6,259372000,24
7,219001013,24
8,259277000,24
9,219016938,24


*Hourly centroid of vessel movement.*

In [14]:
run("""
SELECT MMSI, date_trunc('hour', t) AS ReportHour,
  ST_Centroid(ST_Collect(array_agg(geom))) AS Centroid
FROM AISInput
GROUP BY MMSI, ReportHour ORDER BY MMSI, ReportHour;
""")

,MMSI,ReportHour,Centroid
0,3638,2026-01-20 00:00:00,"b'\x01\x01\x00\x00\x00\xceRR\x13\x03\xee%@8,\x..."
1,3638,2026-01-20 01:00:00,b'\x01\x01\x00\x00\x00\xc4~\xc0\xdb\x03\xee%@\...
2,3638,2026-01-20 02:00:00,"b""\x01\x01\x00\x00\x00\xda\xa90&\x04\xee%@\xd1..."
3,3638,2026-01-20 03:00:00,b'\x01\x01\x00\x00\x00\xdbQ}^\x02\xee%@\xb2\xc...
4,3638,2026-01-20 04:00:00,b'\x01\x01\x00\x00\x00)\xdc\xc5U\x02\xee%@\tk\...
5,3638,2026-01-20 05:00:00,b'\x01\x01\x00\x00\x00\xeeo\xd2\xea\x01\xee%@ ...
6,3638,2026-01-20 06:00:00,b'\x01\x01\x00\x00\x00\xf9\xf0\xe8-\x02\xee%@\...
7,3638,2026-01-20 07:00:00,b'\x01\x01\x00\x00\x00\x15O\xcd~\x02\xee%@e\xb...
8,3638,2026-01-20 08:00:00,b'\x01\x01\x00\x00\x00\x12\xf3<e\x00\xee%@\xf4...
9,3638,2026-01-20 09:00:00,b'\x01\x01\x00\x00\x00\xd3\xf9\x17X\xfc\xed%@\...


*Bounding box of all AIS positions.*

In [15]:
run("""
SELECT MIN(Latitude) AS MinLat, MAX(Latitude) AS MaxLat,
  MIN(Longitude) AS MinLon, MAX(Longitude) AS MaxLon
FROM AISInput;
/* | MinLat    | MaxLat | MinLon    | MaxLon  |
   | -83.4967  |  91    | -115.8718 | 40.3375 | */
""")

,MinLat,MaxLat,MinLon,MaxLon
0,-83.4967,91.0,-115.871855,40.337515


*Ships within 1,000 m of a given ship at a given time, with a margin of one minute.*

In [16]:
run("""
WITH Target AS (
  SELECT Geom, t
  FROM AISInput
  WHERE MMSI = 219015442
  ORDER BY t DESC LIMIT 1 )
SELECT other.MMSI, other.t, ST_AsText(other.Geom) AS Geom
FROM AISInput other, Target t
WHERE other.MMSI <> 219015442 AND
  ST_Distance_Sphere(other.Geom, t.Geom) <= 1000 AND
  abs(EXTRACT(EPOCH FROM (other.t - t.t))) <= 60;
""")

,MMSI,T,Geom
0,219007027,2026-01-20 23:58:39,POINT (10.585922 57.715918)
1,219007027,2026-01-20 23:58:39,POINT (10.585922 57.715918)
2,219020150,2026-01-20 23:58:39,POINT (10.58757 57.715627)
3,219005732,2026-01-20 23:58:40,POINT (10.586303 57.717898)
4,259414000,2026-01-20 23:58:40,POINT (10.592792 57.71838)
5,251323000,2026-01-20 23:58:40,POINT (10.587627 57.716725)
6,265019000,2026-01-20 23:58:40,POINT (10.600387 57.719012)
7,220141000,2026-01-20 23:58:40,POINT (10.595192 57.71787)
8,220141000,2026-01-20 23:58:40,POINT (10.595192 57.71787)
9,220076700,2026-01-20 23:58:41,POINT (10.587072 57.714893)


*For every ship, number of neighbors within 1,000 m considering the ship's latest position (consider space and time, with margin of one minute).*

In [17]:
run("""
WITH Latest AS (
  SELECT MMSI, geom, t
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY MMSI
      ORDER BY t DESC) AS RowNo
    FROM AISInput )
  WHERE RowNo = 1 ),
Neighbors AS (
  SELECT l1.MMSI AS MMSI1, l2.MMSI AS MMSI2
  FROM Latest l1, Latest l2
  WHERE l1.MMSI <> l2.MMSI AND
    ST_Distance_Sphere(l1.Geom, l2.Geom) <= 1000 AND
    abs(EXTRACT(EPOCH FROM (l1.t - l2.t))) <= 60 )
SELECT l.MMSI, ST_AsText(l.Geom) AS Geom, COUNT(n.MMSI2) AS NeighborCount
FROM Latest l LEFT JOIN Neighbors n ON l.MMSI = n.MMSI1
GROUP BY l.MMSI, l.Geom ORDER BY NeighborCount DESC;
""")

,MMSI,Geom,NeighborCount
0,219007727,POINT (9.962617 57.592465),48
1,219005803,POINT (9.96216 57.592787),47
2,219008783,POINT (9.961003 57.59277),47
3,219014692,POINT (10.586925 57.715635),47
4,219007471,POINT (9.964123 57.592407),47
5,219001111,POINT (10.591937 57.718913),47
6,219015595,POINT (9.95605 57.591872),46
7,219015442,POINT (10.58772 57.716918),45
8,219031429,POINT (9.963523 57.592705),45
9,219027752,POINT (9.964912 57.592912),45


*Five nearest neighbors of our target ship considering space and time.*

In [18]:
run("""
WITH Latest AS (
  SELECT MMSI, geom, t
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY MMSI
      ORDER BY t DESC) AS RowNo
    FROM AISInput )
  WHERE RowNo = 1 ),
Target AS ( SELECT MMSI, geom, t FROM Latest WHERE MMSI = 219015442 )
SELECT o.MMSI, ST_AsText(o.geom) AS geom,
  ST_Distance_Sphere(o.geom, t.geom) AS Distance,
  abs(EXTRACT(EPOCH FROM (o.t - t.t))) AS TimeDiff
FROM Latest o JOIN Target t ON o.MMSI <> t.MMSI
WHERE ST_Distance_Sphere(o.geom, t.geom) <= 1000 AND
  abs(EXTRACT(EPOCH FROM (o.t - t.t))) <= 60
ORDER BY Distance LIMIT 5;
/* | MMSI | Geom | Distance | TimeDiff |
   | 251323000 | POINT (10.587627 57.716725) |  22.16 | 59.0 | ...*/
""")

,MMSI,geom,Distance,TimeDiff
0,251323000,POINT (10.587627 57.716725),22.159973,59.0
1,219002827,POINT (10.587328 57.71746),64.607833,18.0
2,220343000,POINT (10.586863 57.717345),69.604797,19.0
3,220333000,POINT (10.587002 57.717613),88.264086,3.0
4,219034309,POINT (10.586427 57.716427),94.221460,18.0


In [19]:
run("""
INSTALL spatial;
INSTALL '/path/to/mobilityduck.duckdb_extension';
LOAD spatial;
LOAD mobilityduck;

CREATE TABLE ships (MMSI integer, Trip tgeompoint, SOG tfloat,  COG tfloat);
INSERT INTO ships
SELECT MMSI, Trip, SOG, COG
FROM postgres_scan('{conn_str}', 'public', 'ships');

SET threads=1;
SET memory_limit='7000MB';

SELECT COUNT(*) FROM ships;
-- 2701

ALTER TABLE ships ADD COLUMN trajectory geometry;
UPDATE ships SET trajectory = trajectory(trip);
CREATE INDEX rtree_trip ON Ships USING RTREE(trajectory);
""")

,Count


## MobilityDuck: AIS trajectories

The queries of Sect. "Querying MobilityDuck" on the table `Ships` of the AIS database of the book, read from PostgreSQL with `postgres_scan`.

*Number of ships at each hour between 1pm and 7pm.*

In [20]:
run("""
WITH Hours AS (
  SELECT TIMESTAMPTZ '2026-01-20 13:00:00+00' AS hour_start
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 14:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 15:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 16:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 17:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 18:00:00+00'
  UNION ALL SELECT TIMESTAMPTZ '2026-01-20 19:00:00+00' )
SELECT h.hour_start, COUNT(DISTINCT s.MMSI) AS num_ships
FROM Hours h, Ships s
WHERE atTime(s.Trip, h.hour_start) IS NOT NULL
GROUP BY h.hour_start ORDER BY h.hour_start;
/* hour_start               num_ships
   2026-01-20 14:00:00+01        1657
   2026-01-20 15:00:00+01        1690
   2026-01-20 16:00:00+01        1657
   2026-01-20 17:00:00+01        1650
   2026-01-20 18:00:00+01        1690
   2026-01-20 19:00:00+01        1711
   2026-01-20 20:00:00+01        1692 */
""")

,hour_start,num_ships
0,2026-01-20 14:00:00+01:00,1657
1,2026-01-20 15:00:00+01:00,1690
2,2026-01-20 16:00:00+01:00,1657
3,2026-01-20 17:00:00+01:00,1650
4,2026-01-20 18:00:00+01:00,1690
5,2026-01-20 19:00:00+01:00,1711
6,2026-01-20 20:00:00+01:00,1692


*Ships within the port of Göteborg.*

In [21]:
run("""
WITH Goteborg(GotEnv) AS (
  SELECT ST_MakeEnvelope(666538, 6392057, 679171, 6403745))
SELECT s.MMSI,
  ST_AsText(trajectory(atGeometry(s.Trip,  g.GotEnv))) AS traj, g.GotEnv
FROM Goteborg g, Ships s
WHERE eIntersects(s.Trip, g.GotEnv);
""")

,MMSI,traj,GotEnv
0,210167000,MULTILINESTRING ((666537.9999955071 6395400.09...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
1,210548000,LINESTRING (666537.9999990496 6395227.61606471...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
2,211839040,MULTILINESTRING ((678604.1785360001 6403536.86...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
3,219003122,GEOMETRYCOLLECTION (MULTIPOINT (675909.8930258...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
4,219011922,LINESTRING (678615.0103369185 6403745.00000054...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
5,219014435,MULTILINESTRING ((670615.166426854 6392895.639...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
6,219020688,LINESTRING (668820.1718878024 6392056.99998675...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
7,219026170,LINESTRING (670590.3506348464 6392778.33383487...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
8,219027463,GEOMETRYCOLLECTION (POINT (666647.9667754653 6...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
9,219029880,GEOMETRYCOLLECTION (POINT (671572.7918987392 6...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...


*Compute and list the vessel's trajectories between the ports of R{\o*

In [22]:
run("""
WITH Ports(Rodby, Puttgarden) AS (
  SELECT ST_MakeEnvelope(651135, 6058230, 651422, 6058548),
    ST_MakeEnvelope(644339, 6042108, 644896, 6042487) )
SELECT s.MMSI, ST_AsText(trajectory(trip)) AS Trajectory, Rodby, Puttgarden
FROM Ports p, Ships s
WHERE eIntersects(trip, p.Rodby) AND eIntersects(trip, p.Puttgarden);
""")

,MMSI,Trajectory,Rodby,Puttgarden
0,211190000,LINESTRING (646261.7990774247 6046358.79709851...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
1,211757540,LINESTRING (644418.9075008206 6041768.43210913...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
2,219000429,LINESTRING (650530.3215489659 6057549.03231307...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...
3,219000431,LINESTRING (644884.6442705076 6044842.92779543...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...


*Compute and display the vessel activity between the ports of R{\o*

In [23]:
run("""
WITH AlertBelt(Belt) AS (
  SELECT ST_MakeEnvelope(640730, 6058230, 654100, 6042487))
SELECT MMSI, atGeometry(s.Trip, b.Belt),
  ST_AsText(trajectory(atGeometry(s.Trip,  b.Belt))) AS traj
FROM ships s, AlertBelt b
WHERE eIntersects(s.Trip, b.Belt);
""")

,MMSI,"atgeometry(s.Trip, b.Belt)",traj
0,205196000,b'`k\x02\x00.\x03^\x00\x01\x00\x00\x00\xc0\x02...,LINESTRING (654084.9382176222 6056936.27177283...
1,205210000,b'\xc0\x8f\x1d\x00.\x03^\x00\x01\x00\x00\x00\x...,LINESTRING (650646.3844959944 6052875.02699513...
2,205214000,b'\xa0\xc3\x04\x00.\x03^\x00\x01\x00\x00\x00n\...,LINESTRING (654062.5318483934 6057086.08091990...
3,209415000,b' \xde\x00\x00.\x03^\x00\x01\x00\x00\x00\xfa\...,LINESTRING (640729.999995506 6050149.311300968...
4,209731000,b'\xa0\xef\x00\x00.\x03^\x00\x01\x00\x00\x00\x...,LINESTRING (640729.9999975422 6050511.28309092...
5,209982000,b'\x00\xf4\x00\x00.\x03^\x00\x01\x00\x00\x00\x...,LINESTRING (640729.9999986589 6050145.57074222...
6,210359000,b'@\x00\x01\x00.\x03^\x00\x01\x00\x00\x00!\x01...,LINESTRING (640729.9999972438 6050139.08891299...
7,210665000,b'@\xb3\x00\x00.\x03^\x00\x01\x00\x00\x00\xc9\...,LINESTRING (654100.000001139 6047238.452950414...
8,210731000,b'\x00\xed\x00\x00.\x03^\x00\x01\x00\x00\x00\x...,LINESTRING (654100.0000039468 6046451.92771392...
9,211130000,b'@4\x02\x00.\x03^\x00\x02\x00\x00\x00\x7f\x02...,MULTILINESTRING ((650881.1776369439 6043007.69...


*Compute and display the vessels that during a time interval have been at 300m from each other, displaying the points, the closest distance, and the position of the vessels.*

In [24]:
run("""
WITH Timespan(Period) AS (
  SELECT tstzspan '[2026-01-20 11:00:00+00, 2026-01-20 23:30:00+00)' ),
TimeShips(MMSI, Trip) AS (
  SELECT MMSI, atTime(s.Trip, t.Period)
  FROM Ships s TABLESAMPLE RESERVOIR (300 ROWS) REPEATABLE (1),
    Timespan t
  WHERE atTime(s.Trip, t.Period) IS NOT NULL ),
TimeClosestShips(MMSI1, MMSI2, ClosestDist, TimeAtClosestDist, Trip1, Trip2) AS (
  SELECT s1.MMSI, s2.MMSI, startValue(atMin(s1.Trip <-> s2.Trip)),
    startTimestamp(atMin(s1.Trip <-> s2.Trip)), s1.Trip, s2.Trip
  FROM TimeShips s1, TimeShips s2
  WHERE s1.MMSI < s2.MMSI AND CASE WHEN eDwithin(s1.Trip, s2.Trip, 300)
    THEN startValue(atMin(s1.Trip <-> s2.Trip)) < 300 END )
SELECT t.MMSI1, t.MMSI2, t.ClosestDist, t.TimeAtClosestDist,
  valueAtTimestamp(Trip1, TimeAtClosestDist) AS Point1,
  valueAtTimestamp(Trip2, TimeAtClosestDist) AS Point2
FROM TimeClosestShips t;
""")

,MMSI1,MMSI2,ClosestDist,TimeAtClosestDist,Point1,Point2
0,211176000,211283830,222.313401,2026-01-20 22:19:15+01:00,b'\x01\x01\x00\x00\x00\xc0\xe3:\xb3\xaeK\x19AD...,b'\x01\x01\x00\x00\x00\xc8[\xd4\x98\xf1M\x19A\...
1,211222750,211332370,273.739213,2026-01-20 16:31:10+01:00,"b'\x01\x01\x00\x00\x00\xad\x96J""\xc6]\x1dA\x13...","b'\x01\x01\x00\x00\x00\x18""\x02\x08\xd4Y\x1dA\..."
2,211222750,211429600,36.648829,2026-01-20 16:32:25.631258+01:00,b'\x01\x01\x00\x00\x00c\xfb0fd^\x1dAR\xb6r\xc8...,b'\x01\x01\x00\x00\x00*9\xaf9\xf3^\x1dA\xb5/\x...
3,211222750,211448440,35.063658,2026-01-20 16:34:08.007255+01:00,"b""\x01\x01\x00\x00\x00\xb2'M\x80\xbea\x1dA\x97...",b'\x01\x01\x00\x00\x00nN\xd1s\x06b\x1dA\xe2HnK...
4,211332370,211448440,103.651814,2026-01-20 21:18:30+01:00,b'\x01\x01\x00\x00\x00\xd3\xf7\xe3\xbf\xd6Y\x1...,b'\x01\x01\x00\x00\x00\xc5O\xd1EmY\x1dA\x83N\x...
5,211429600,211448440,86.393707,2026-01-20 14:59:58+01:00,b'\x01\x01\x00\x00\x00pT\x04\x8a\n_\x1dA\xec\x...,b'\x01\x01\x00\x00\x00\xa4m\xb5XP`\x1dA\x00(\x...
6,211279690,211795250,216.413733,2026-01-20 23:36:32+01:00,b'\x01\x01\x00\x00\x00\xc5>\x87\x8f\x08J A\xe3...,b'\x01\x01\x00\x00\x00.-\x96\xcc\xa7I A\xcf\xe...
7,211182000,211796430,77.525420,2026-01-20 21:54:25+01:00,b'\x01\x01\x00\x00\x00c\x08\xcbc\x8fF\x1dABZ \...,b'\x01\x01\x00\x00\x00^@K\t\xc5G\x1dA\x9e\xc9\...
8,211315620,211815000,262.183533,2026-01-20 13:43:38+01:00,b'\x01\x01\x00\x00\x00\xb2\xa8\xa4\x03\xe0\xf7...,b'\x01\x01\x00\x00\x00\xcc\xcdSi\xb9\xf7\x1dA\...
9,211749880,211833390,257.379020,2026-01-20 15:07:44+01:00,b'\x01\x01\x00\x00\x00\xbd\x9f0{4r%A\xa3Jw\x17...,b'\x01\x01\x00\x00\x00\xccI\xd3.7p%A\xe6)8p\xd...


*Compute and display the trajectories*

In [25]:
run("""
WITH Timespan(Period) AS (
  SELECT tstzspan '[2026-01-20 11:00:00+00, 2026-01-20 23:30:00+00)' ),
TimeShips(MMSI, Trip) AS (
  SELECT MMSI, atTime(s.Trip, t.Period)
  FROM Ships s TABLESAMPLE RESERVOIR (300 ROWS) REPEATABLE (1),
    Timespan t
  WHERE atTime(s.Trip, t.Period) IS NOT NULL )
SELECT s1.MMSI, s2.MMSI, s1.Trip, s2.Trip, ST_AsText(trajectory(s1.Trip)) AS traj1,
  ST_AsText(trajectory(s2.Trip)) AS traj2
FROM TimeShips s1, TimeShips s2
WHERE s1.MMSI < s2.MMSI AND eDwithin(s1.Trip, s2.Trip, 300)  ;
""")

,MMSI,MMSI,Trip,Trip,traj1,traj2
0,211176000,211283830,b'`\n\x00\x00.\x03^\x00\x03\x00\x00\x00\x04\x0...,b'\x00\x1b\x00\x00.\x03^\x00\x03\x00\x00\x00\x...,GEOMETRYCOLLECTION (MULTIPOINT (414441.5107486...,GEOMETRYCOLLECTION (POINT (414588.707223943 59...
1,211310410,211315620,b'\xa0\\\x00\x00.\x03^\x00\x17\x00\x00\x00:\x0...,b'\xc0\xa8\x00\x00.\x03^\x00\x11\x00\x00\x00\x...,GEOMETRYCOLLECTION (MULTIPOINT (490930.7645944...,GEOMETRYCOLLECTION (MULTIPOINT (491000.0085419...
2,211222750,211332370,b'\xa09\x00\x00.\x03^\x00\x07\x00\x00\x002\x00...,b'\xc0b\x00\x00.\x03^\x00\x12\x00\x00\x00K\x00...,GEOMETRYCOLLECTION (MULTIPOINT (480963.5806333...,GEOMETRYCOLLECTION (MULTIPOINT (480885.0838749...
3,211222750,211429600,b'\xa09\x00\x00.\x03^\x00\x07\x00\x00\x002\x00...,b'``\x03\x00.\x03^\x00\x08\x00\x00\x00\xca\x03...,GEOMETRYCOLLECTION (MULTIPOINT (480963.5806333...,MULTILINESTRING ((481211.229949662 5968435.139...
4,211222750,211448440,b'\xa09\x00\x00.\x03^\x00\x07\x00\x00\x002\x00...,b'\xe0\xcc\x03\x00.\x03^\x00\x08\x00\x00\x00F\...,GEOMETRYCOLLECTION (MULTIPOINT (480963.5806333...,MULTILINESTRING ((481017.0396857883 5969073.42...
5,211332370,211448440,b'\xc0b\x00\x00.\x03^\x00\x12\x00\x00\x00K\x00...,b'\xe0\xcc\x03\x00.\x03^\x00\x08\x00\x00\x00F\...,GEOMETRYCOLLECTION (MULTIPOINT (480885.0838749...,MULTILINESTRING ((481017.0396857883 5969073.42...
6,211429600,211448440,b'``\x03\x00.\x03^\x00\x08\x00\x00\x00\xca\x03...,b'\xe0\xcc\x03\x00.\x03^\x00\x08\x00\x00\x00F\...,MULTILINESTRING ((481211.229949662 5968435.139...,MULTILINESTRING ((481017.0396857883 5969073.42...
7,211279690,211795250,"b'\x80,\x00\x00.\x03^\x00\x07\x00\x00\x00#\x00...",b'@\xb8\x04\x00.\x03^\x00\n\x00\x00\x00O\x05\x...,GEOMETRYCOLLECTION (POINT (538283.2298427728 5...,GEOMETRYCOLLECTION (POINT (533709.4812504792 5...
8,211182000,211796430,b'\xc0F\x00\x00.\x03^\x00\x10\x00\x00\x00/\x00...,b'`e\x0e\x00.\x03^\x00\x01\x00\x00\x00p\x10\x0...,GEOMETRYCOLLECTION (MULTIPOINT (479651.1568326...,LINESTRING (486429.616656115 6050823.502134861...
9,211315620,211815000,b'\xc0\xa8\x00\x00.\x03^\x00\x11\x00\x00\x00\x...,b'\x000\x00\x00.\x03^\x00\x0b\x00\x00\x00\x1f\...,GEOMETRYCOLLECTION (MULTIPOINT (491000.0085419...,GEOMETRYCOLLECTION (MULTIPOINT (490980.9614875...


*Trajectories passing through a buffer with a radius of 500m around the port of Copenhagen.*

In [26]:
run("""
WITH Port AS (
  SELECT ST_Point(726765.3929300838, 6178705.65508127) AS PortGeom ),
PortBuffer AS ( SELECT ST_Buffer(PortGeom, 500) AS buf FROM Port )
SELECT s.MMSI, ST_AsText(trajectory(s.Trip)) AS TripWkt,
  ST_AsWKB(pb.buf) AS BufferWkb
FROM Ships s, PortBuffer pb
WHERE eIntersects(s.Trip, pb.buf);
""")

,MMSI,TripWkt,BufferWkb
0,219002798,LINESTRING (727707.1985358351 6177304.80292357...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...
1,219024675,LINESTRING (732793.6246569207 6165408.62672716...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...
2,219385000,LINESTRING (725887.4127071316 6176231.61954977...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...


*Trajectories that stayed within radius of 1000m around the port of Copenhagen for at least twenty minutes.*

In [27]:
run("""
WITH Port AS (
  SELECT ST_Point(726765.3929300838, 6178705.65508127) AS PortGeom ),
PortBuffer AS ( SELECT ST_Buffer(PortGeom, 1000) AS buf FROM Port )
SELECT s.MMSI,
  asText(stops(atGeometry(s.trip, pb.buf),10.0, INTERVAL '20 minutes')),
  ST_AsText(trajectory(atGeometry(s.trip, pb.buf))) AS tripWkt,
  ST_AsWKB(pb.buf) AS bufferWkb
FROM ships s, PortBuffer pb
WHERE eIntersects(s.trip, pb.buf) AND
  astext(stops(atGeometry(s.trip, pb.buf),10.0, INTERVAL '20 minutes')) IS NOT NULL;
""")

,MMSI,"astext(stops(atgeometry(s.trip, pb.buf), 10.0, CAST('20 minutes' AS INTERVAL)))",tripWkt,bufferWkb
0,219001362,{[POINT(726975.5530205681 6179514.156220158)@2...,MULTILINESTRING ((727091.316211704 6179646.752...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
1,219002798,{[POINT(727336.4119952423 6179379.998396069)@2...,MULTILINESTRING ((727130.8289535276 6179634.76...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
2,219002801,{[POINT(726928.247745734 6179491.616234032)@20...,MULTILINESTRING ((727135.1340101883 6179633.46...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
3,219003121,{[POINT(727372.28430405 6179375.516463043)@202...,LINESTRING (727374.782006778 6179372.301548108...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
4,219025528,{[POINT(726928.18481275 6179415.560635618)@202...,MULTILINESTRING ((727131.6033996923 6179634.53...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."
5,219031423,{[POINT(726251.2063936263 6179537.872890741)@2...,LINESTRING (726251.2063936263 6179537.87289074...,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0..."


*Trajectories within a radius of 1000m around the port of Malmoö, for at least one hour. In this case, consider the port of Malmö as an area with a radius of 5000m from the point with coordinates (752775.913, 6167956.005, SRID = 25832). Also compute the start instant of this stop.*

In [28]:
run("""
WITH Port AS (
  SELECT ST_Point(752775.9137447166, 6167956.005944439) AS PortGeom ),
PortBuffer AS ( SELECT ST_Buffer(PortGeom, 5000) AS buf FROM Port )
SELECT s.MMSI, asText(stops(atGeometry(s.trip, pb.buf),1000.0, INTERVAL '1 hour')),
  ST_AsText(trajectory(atGeometry(s.trip, pb.buf))) AS TripWkt,
  ST_AsWKB(pb.buf) AS BufferWkb,
  startTimestamp(stops(atGeometry(s.trip, pb.buf),1000.0,
  INTERVAL '1 hour')) AS StartTime
FROM Ships s, PortBuffer pb
WHERE eIntersects(s.trip, pb.buf) AND
  astext(stops(atGeometry(s.trip, pb.buf),1000.0, INTERVAL '1 hour')) IS NOT NULL;
""")

,MMSI,"astext(stops(atgeometry(s.trip, pb.buf), 1000.0, CAST('1 hour' AS INTERVAL)))",TripWkt,BufferWkb,StartTime
0,212797000,{[POINT(752635.6332655648 6172942.189523194)@2...,MULTILINESTRING ((752639.0154889377 6172940.01...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 18:23:53.108025+01:00
1,219027899,{[POINT(751211.0599393302 6170907.990429523)@2...,LINESTRING (750235.3507725103 6172240.18719179...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 16:45:57+01:00
2,219159000,{[POINT(751233.5084446241 6170938.06546821)@20...,MULTILINESTRING ((750167.4360563783 6172203.88...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 04:27:53+01:00
3,232059804,{[POINT(753007.2855194729 6171611.757306266)@2...,LINESTRING (750139.2598362201 6172188.82551294...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 08:24:59+01:00
4,255739000,{[POINT(751130.9699972508 6171034.799380907)@2...,LINESTRING (750149.364712106 6172194.226685133...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 12:32:40+01:00
5,265042000,{[POINT(750211.1242940824 6172023.921071003)@2...,LINESTRING (750050.5282879829 6172141.39751698...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 18:42:23+01:00
6,265514800,{[POINT(751714.0670077421 6170332.867291322)@2...,LINESTRING (751714.0670077421 6170332.86729132...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 01:00:04+01:00
7,265594380,{[POINT(753936.2420363978 6171684.916205654)@2...,LINESTRING (753936.2420363978 6171684.91620565...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 10:20:38+01:00
8,265607640,{[POINT(753976.3883683694 6171675.203821069)@2...,LINESTRING (753976.3883683694 6171675.20382106...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 01:01:16+01:00
9,265610950,{[POINT(750924.7417881248 6170938.931090504)@2...,MULTILINESTRING ((751705.7607960323 6170343.32...,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00!\x00\x0...,2026-01-20 02:51:59+01:00
